In [1]:
import sys 
if ".." not in sys.path:
    sys.path.append("..")

import os
from Bio import SeqIO
import pandas as pd
from multiprocessing import Pool
from Metrics.utils import calculate_rmsd, calculate_pLDDT, run_dockq
from Metrics.utils import calculate_tcr_cdr_rmsd, calculate_tcr_cdr3ab_rmsd
from Metrics.utils import calculate_tcr_cdr_pLDDT, calculate_tcr_cdr3ab_pLDDT

In [2]:
data_path = '../../benchmark_data'
tcr_pmhci_df = pd.read_csv(os.path.join(data_path, 'tcr-pmhc_info.csv'), sep=',', header=0)
tcr_pmhci_df = tcr_pmhci_df[tcr_pmhci_df['MHC Class']=='I']
print(len(tcr_pmhci_df))

cutoff_date = '2021-09-30'
tcr_pmhci_df['Release Date'] = pd.to_datetime(tcr_pmhci_df['Release Date'])
df_sorted = tcr_pmhci_df.sort_values(by='Release Date')
after_cutoff_ids = df_sorted[df_sorted['Release Date'] > cutoff_date]["PDB ID"].tolist()
after_cutoff_ids.sort()

resolution = "CA"        # "backbone"

fasta_path = os.path.join(data_path, 'class-i/fasta')
true_pdb_dir = os.path.join(data_path, 'class-i/pdb')
pred_pdb_dir = '../pred_pdb/protenix_v1/pmhc_tcr/classI/classI_protenix_v1_no_msa'

result_dir = f'../eval_results_{resolution}/tcr-pmhci'
os.makedirs(result_dir, exist_ok=True)

save_filename = '_'.join(pred_pdb_dir.split('/')[-2:-1]) + '.csv'
result_path = os.path.join(result_dir, save_filename)
print(f"Result will be saved to {result_path}")

56
Result will be saved to ../eval_results_CA/tcr-pmhci/classI.csv


In [3]:
id2seq_dict = {}

for id in after_cutoff_ids:
    seqs = {}
    seqs['cdr1a'] = tcr_pmhci_df[tcr_pmhci_df['PDB ID']==id]['TRA CDR1'].item()
    seqs['cdr2a'] = tcr_pmhci_df[tcr_pmhci_df['PDB ID']==id]['TRA CDR2'].item()
    seqs['cdr3a'] = tcr_pmhci_df[tcr_pmhci_df['PDB ID']==id]['TRA CDR3'].item()
    
    seqs['cdr1b'] = tcr_pmhci_df[tcr_pmhci_df['PDB ID']==id]['TRB CDR1'].item()
    seqs['cdr2b'] = tcr_pmhci_df[tcr_pmhci_df['PDB ID']==id]['TRB CDR2'].item()
    seqs['cdr3b'] = tcr_pmhci_df[tcr_pmhci_df['PDB ID']==id]['TRB CDR3'].item()

    fasta_file = os.path.join(fasta_path, f"{id}_pmhc_tcr.fasta")
    for record in SeqIO.parse(fasta_file, "fasta"):
        if record.id == "Chain_C":
            seqs['tcra'] = str(record.seq)
        elif record.id == "Chain_D":
            seqs['tcrb'] = str(record.seq)
    
    if seqs['cdr1a'] not in seqs['tcra']:
        print(id)
    if seqs['cdr2a'] not in seqs['tcra']:
        print(id)
    if seqs['cdr3a'] not in seqs['tcra']:
        print(id)
    if seqs['cdr1b'] not in seqs['tcrb']:
        print(id)
    if seqs['cdr2b'] not in seqs['tcrb']:
        print(id)
    if seqs['cdr3b'] not in seqs['tcrb']:
        print(id)

    seqs['pdb_type'] = 'AB'
    id2seq_dict[id] = seqs

7rm4
7rm4
7rm4
7rm4
7rm4
7rm4


In [4]:
tmp = id2seq_dict['7rm4']['tcra']
id2seq_dict['7rm4']['tcra'] = id2seq_dict['7rm4']['tcrb']
id2seq_dict['7rm4']['tcrb'] = tmp
id2seq_dict['7rm4']['pdb_type'] = 'BA'

id2seq_dict['7rm4']

{'cdr1a': 'NYTNYSPAYLQ',
 'cdr2a': 'LLIRENEKEK',
 'cdr3a': 'CALDIYPHDMRF',
 'cdr1b': 'NPISGHATLY',
 'cdr2b': 'IQFQNNGVVDD',
 'cdr3b': 'CASSLDPGDTGELFF',
 'tcra': 'IEQNSEALNIQEGKTATLTCNYTNYSPAYLQWYRQDPGRGPVFLLLIRENEKEKRKERLKVTFDTTLKQSLFHITASQPADSATYLCALDIYPHDMRFGAGTRLTVK',
 'tcrb': 'VAQSPRYKIIEKRQSVAFWCNPISGHATLYWYQQILGQGPKLLIQFQNNGVVDDSQLPKDRFSAERLKGVDSTLKIQPAKLEDSAVYLCASSLDPGDTGELFFGEGSRLTVL',
 'pdb_type': 'BA'}

## Create result dataframe

In [5]:
def get_id2true_pred_dict(true_pdb_dir, pred_pdb_dir, after_cutoff_ids):
    true_pdb_files = [f for f in os.listdir(true_pdb_dir) if f.endswith('.pdb')]
    pred_pdb_files = [f for f in os.listdir(pred_pdb_dir) if f.endswith('.pdb')]

    id2true_pred_dict = {}
    for id in after_cutoff_ids:
        true_pred_dict = {}
        for f in true_pdb_files:
            if (id in f):
                true_pred_dict['true_pdb_path'] = os.path.join(true_pdb_dir, f)
                break
        for f in pred_pdb_files:
            if (id in f):
                true_pred_dict['pred_pdb_path'] = os.path.join(pred_pdb_dir, f)
                break
        if len(true_pred_dict.keys()) == 2:
            true_pred_dict[cutoff_date] = 'after' if id in after_cutoff_ids else 'before'
            id2true_pred_dict[id] = true_pred_dict
        else:
            print(f"{id} only has {true_pred_dict.keys()}")     # check missing one
    return id2true_pred_dict

In [6]:
def calculate_rmsds(k, v, resolution):
    rmsds = {
        'RMSD': calculate_rmsd(v['pred_pdb_path'], v['true_pdb_path'], atom_type=resolution), 
        'RMSD-mhc': calculate_rmsd(v['pred_pdb_path'], v['true_pdb_path'], atom_type=resolution, chains=['A']), 
        'RMSD-pep': calculate_rmsd(v['pred_pdb_path'], v['true_pdb_path'], atom_type=resolution, chains=['B']), 
        'RMSD-pmhc': calculate_rmsd(v['pred_pdb_path'], v['true_pdb_path'], atom_type=resolution, chains=['A', 'B']), 
        'RMSD-tcr': calculate_rmsd(v['pred_pdb_path'], v['true_pdb_path'], atom_type=resolution, chains=['C', 'D']), 
    }
    print(f"{k}: {rmsds}")
    return k, rmsds

def add_rmsds_to_dict(id2true_pred_dict, num_processes):
    # Prepare the data for parallel processing
    items = list(id2true_pred_dict.items())

    # Use multiprocessing to compute RMSD values
    with Pool(processes=num_processes) as pool:
        # Apply the RMSD calculation function in parallel
        results = pool.starmap(calculate_rmsds, [(k, v, resolution) for k, v in items])

    return {k: metrics for k, metrics in results if metrics is not None}


def calculate_cdr_rmsds(k, v, seq_dict, resolution):
    cdr_rmsds = calculate_tcr_cdr_rmsd(v['pred_pdb_path'], v['true_pdb_path'], seq_dict, 
                                        atom_type=resolution, chain_ids=['C', 'D'])
    cdr_rmsds.update(calculate_tcr_cdr3ab_rmsd(v['pred_pdb_path'], v['true_pdb_path'], seq_dict, 
                                        atom_type=resolution, chain_ids=['C', 'D']))
    print(f"{k}: {cdr_rmsds}")
    return k, cdr_rmsds

def add_cdr_rmsds_to_dict(id2true_pred_dict, num_processes):
    # Prepare the data for parallel processing
    items = list(id2true_pred_dict.items())

    # Use multiprocessing to compute RMSD values
    with Pool(processes=num_processes) as pool:
        # Apply the RMSD calculation function in parallel
        results = pool.starmap(calculate_cdr_rmsds, [(k, v, id2seq_dict[k], resolution) for k, v in items])

    return {k: metrics for k, metrics in results if metrics is not None}


def calculate_plddts(k, v, resolution):
    plddts = {
        'pLDDT': calculate_pLDDT(v['pred_pdb_path'], atom_type=resolution), 
        'pLDDT-mhc': calculate_pLDDT(v['pred_pdb_path'], atom_type=resolution, chains=['A']), 
        'pLDDT-pep': calculate_pLDDT(v['pred_pdb_path'], atom_type=resolution, chains=['B']), 
        'pLDDT-pmhc': calculate_pLDDT(v['pred_pdb_path'], atom_type=resolution, chains=['A', 'B']), 
        'pLDDT-tcr': calculate_pLDDT(v['pred_pdb_path'], atom_type=resolution, chains=['C', 'D']), 
    }
    print(f"{k}: {plddts}")
    return k, plddts

def add_plddts_to_dict(id2true_pred_dict, num_processes):
    # Prepare the data for parallel processing
    items = list(id2true_pred_dict.items())

    # Use multiprocessing to compute RMSD values
    with Pool(processes=num_processes) as pool:
        # Apply the RMSD calculation function in parallel
        results = pool.starmap(calculate_plddts, [(k, v, resolution) for k, v in items])

    return {k: metrics for k, metrics in results if metrics is not None}


def calculate_cdr_plddts(k, v, seq_dict, resolution):
    cdr_plddts = calculate_tcr_cdr_pLDDT(v['pred_pdb_path'], seq_dict, 
                                        atom_type=resolution, chain_ids=['C', 'D'])
    cdr_plddts.update(calculate_tcr_cdr3ab_pLDDT(v['pred_pdb_path'], seq_dict, 
                                        atom_type=resolution, chain_ids=['C', 'D']))
    print(f"{k}: {cdr_plddts}")
    return k, cdr_plddts

def add_cdr_plddts_to_dict(id2true_pred_dict, num_processes):
    # Prepare the data for parallel processing
    items = list(id2true_pred_dict.items())

    # Use multiprocessing to compute RMSD values
    with Pool(processes=num_processes) as pool:
        # Apply the pLDDT calculation function in parallel
        results = pool.starmap(calculate_cdr_plddts, [(k, v, id2seq_dict[k], resolution) for k, v in items])

    return {k: metrics for k, metrics in results if metrics is not None}


def add_dockq_to_dict(id2true_pred_dict, num_processes, cmd_template):
    # Prepare the data for parallel processing
    items = list(id2true_pred_dict.items())
    
    # Use multiprocessing to compute DockQ
    with Pool(processes=num_processes) as pool:
        results = pool.starmap(run_dockq, [(k, v, cmd_template) for k, v in items])
    
    return {
        k: metrics for k, metrics in results if metrics is not None
    }

In [7]:
ranks = ['ranked_0', 'ranked_1', 'ranked_2', 'ranked_3', 'ranked_4']

result_dfs = []
for ranked_i in ranks:
    ranked_i_pred_pdb_dir = os.path.join(pred_pdb_dir, ranked_i)
    
    # Collect pdb files
    id2true_pred_dict = get_id2true_pred_dict(true_pdb_dir, ranked_i_pred_pdb_dir, after_cutoff_ids)
    print(f'Number of pdbids: {len(id2true_pred_dict)}')
    result_df = pd.DataFrame(id2true_pred_dict).T

    # Calculate r.m.s.d
    id2rmsds_dict = add_rmsds_to_dict(id2true_pred_dict, num_processes=16)
    result_df = pd.DataFrame(id2rmsds_dict).T.join(result_df)

    ## Calculate CDR r.m.s.d
    id2cdr_rmsds_dict = add_cdr_rmsds_to_dict(id2true_pred_dict, num_processes=16)
    result_df = pd.DataFrame(id2cdr_rmsds_dict).T.join(result_df)

    # Calculate pLDDT
    id2plddts_dict = add_plddts_to_dict(id2true_pred_dict, num_processes=16)
    result_df = pd.DataFrame(id2plddts_dict).T.join(result_df)

    ## Calculate CDR pLDDT
    id2cdr_plddts_dict = add_cdr_plddts_to_dict(id2true_pred_dict, num_processes=16)
    result_df = pd.DataFrame(id2cdr_plddts_dict).T.join(result_df)

    # Calculate DockQ
    DockQ_path = "../Metrics/DockQ/DockQ.py"
    cmd_template = (
        f"python {DockQ_path} "
        "{pred_path} {true_path} "
        "-short -no_needle "
        f"{'-useCA ' if resolution == 'CA' else ''}" 
        "-native_chain1 A B -model_chain1 A B -perm1 "
        "-native_chain2 C D -model_chain2 C D -perm2" 
    )

    id2dockq_dict = add_dockq_to_dict(id2true_pred_dict, num_processes=16, cmd_template=cmd_template)
    result_df = pd.DataFrame(id2dockq_dict).T.join(result_df)

    result_df['rank'] = ranked_i
    result_df.reset_index(inplace=True)
    result_df.rename(columns={'index':'id'}, inplace=True)

    result_dfs.append(result_df)

# Save result
if os.path.exists(result_path):
    print(f"Rewrite {result_path}")

merged_result_df = pd.concat(result_dfs, axis=0, ignore_index=True)
merged_result_df.to_csv(result_path, sep='\t')

Number of pdbids: 56


6zkz: {'RMSD': np.float32(20.8881), 'RMSD-mhc': np.float32(11.592639), 'RMSD-pep': np.float32(2.3466978), 'RMSD-pmhc': np.float32(11.837811), 'RMSD-tcr': np.float32(16.297415)}
6zkx: {'RMSD': np.float32(22.161951), 'RMSD-mhc': np.float32(15.788837), 'RMSD-pep': np.float32(2.5134861), 'RMSD-pmhc': np.float32(16.319971), 'RMSD-tcr': np.float32(16.272066)}
7n2p: {'RMSD': np.float32(21.14908), 'RMSD-mhc': np.float32(18.71997), 'RMSD-pep': np.float32(2.1993053), 'RMSD-pmhc': np.float32(18.597282), 'RMSD-tcr': np.float32(15.011065)}
7l1d: {'RMSD': np.float32(21.382246), 'RMSD-mhc': np.float32(13.556553), 'RMSD-pep': np.float32(1.8038424), 'RMSD-pmhc': np.float32(17.784006), 'RMSD-tcr': np.float32(15.222848)}7dzm: {'RMSD': np.float32(21.303988), 'RMSD-mhc': np.float32(19.101978), 'RMSD-pep': np.float32(4.177733), 'RMSD-pmhc': np.float32(19.097351), 'RMSD-tcr': np.float32(16.374573)}

6zkw: {'RMSD': np.float32(22.274933), 'RMSD-mhc': np.float32(17.826433), 'RMSD-pep': np.float32(1.894507), 'RM